In [ ]:
import skimage
import numpy as np
import matplotlib.pyplot as plt
import hyperspy.api as hs
import sys
sys.path.append('..')
import roi_tools

s = hs.load('../data/images/Jaume LFO/HAADF_Buena.dm3')
left_bound = 20
right_bound = 2048-20
start_pixel = 200
end_pixel = 2048-100

# s = hs.load('../data/images/Jaume LFO EELS/EEL SI 22 13/Integrated Shifted Spectrum Image adf.dm3')
# left_bound = 0 # TUNE THIS
# right_bound = 185 # TUNE THIS
# start_pixel = 20 # TUNE THIS
# end_pixel = 235 # TUNE THIS

roi = roi_tools.ROI(s, left_bound, right_bound, start_pixel, end_pixel)
roi.build_grid_dict()
roi.get_atom_types()

lu_mask = np.zeros_like(roi.grid, dtype=bool)
for (i, j), patch in np.ndenumerate(roi.grid):
    if patch is not None and getattr(patch, 'atom_type', None) == 'Lu':
        lu_mask[i, j] = True

In [ ]:
min14 = 0.971371
min8 = 0.974433
min4 = 0.975496

# 14
dis = np.array([0, 0, -1, -1, -1, 1, 1, 1, -2, -2, -2, 2, 2, 2])
djs = np.array([-1, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1, -1, 0, 1])
djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity()
roi.get_relative_vicinity()

raw_higher_map_14 = roi.relative_vicinity.astype(float) < min14
higher_map_14 = raw_higher_map_14 & lu_mask
relative_vicinity_14 = roi.relative_vicinity.astype(float)

# 8
dis = np.array([0, 0, -1, -1, -1, 1, 1, 1])
djs = np.array([-1, 1, -1, 0, 1, -1, 0, 1])
djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity()
roi.get_relative_vicinity()

raw_higher_map_8 = roi.relative_vicinity.astype(float) < min8
higher_map_8 = raw_higher_map_8 & lu_mask
relative_vicinity_8 = roi.relative_vicinity.astype(float)

# 4
dis = np.array([0, 0, -1, 1])
djs = np.array([-1, 1, 0, 0])
djs = djs * 2
roi.set_vicinity_coords(dis, djs)
roi.get_vicinity()
roi.get_relative_vicinity()

raw_higher_map_4 = roi.relative_vicinity.astype(float) < min4
higher_map_4 = raw_higher_map_4 & lu_mask
relative_vicinity_4 = roi.relative_vicinity.astype(float)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def plot_vicinity_comparison(roi, lu_mask, configs):
    """
    Generates a three-panel horizontal histogram plot where subplots are adjacent.
    Bars above the simulation cutoff are highlighted in a different color.
    """
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial'],
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'axes.labelsize': 18,     
        'xtick.labelsize': 14,    
        'ytick.labelsize': 14,
        'text.usetex': False 
    })

    fig, axes = plt.subplots(1, 3, figsize=(14, 6), sharey=True)
    
    color_pristine_hist = '#ff7f0e'
    color_defect_hist = '#D6204E' 
    color_pristine_stat = "#da6c0c"
    color_defect_stat = "#9D0229"
    bin_width = 0.0015
    bins = np.arange(0.9, 1.1 + bin_width, bin_width)

    for ax, (label, data_array, cutoff) in zip(axes, configs):
        data = data_array[lu_mask].astype(float)
        data = data[~np.isnan(data)]
        mean_val = np.mean(data)
        
        # Calculate histogram manually to control individual bar colors
        counts, edges = np.histogram(data, bins=bins)
        centers = (edges[:-1] + edges[1:]) / 2
        
        for count, center in zip(counts, centers):
            bar_color = color_defect_hist if center < cutoff else color_pristine_hist
            ax.barh(center, count, height=bin_width, color=bar_color, alpha=0.7, zorder=3)
        
        ax.axhline(y=mean_val, color=color_pristine_stat, linestyle='--', linewidth=1.2, zorder=4)
        ax.axhline(y=cutoff, color=color_pristine_stat, linestyle='-', linewidth=2, zorder=4) 

        trans = ax.get_yaxis_transform()
        ax.text(0.98, cutoff + 0.0005, f'{cutoff:.4f}', color=color_pristine_stat, transform=trans, 
                va='bottom', ha='right', fontsize=14, fontweight='bold', zorder=5)
        ax.text(0.98, mean_val + 0.0005, f'{mean_val:.4f}', color=color_pristine_stat, transform=trans, 
                va='bottom', ha='right', fontsize=14, zorder=5)

        ax.set_title(f'{label} Atoms', fontsize=18, pad=10) 
        ax.tick_params(direction='in', top=False, right=False, length=6)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4, integer=True, prune='upper'))
        ax.set_xlim(0, 55)

    # Proxy artists for legend since we plotted bars individually
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color_defect_hist, alpha=0.7, label='Potential Antisites'),
        Patch(facecolor=color_pristine_hist, alpha=0.7, label='Potential Non-Antisites'),
        Line2D([0], [0], color=color_pristine_stat, linestyle='-', label='Simulation Cutoff', linewidth=2),
        Line2D([0], [0], color=color_pristine_stat, linestyle='--', label='Mean')
    ]

    axes[1].set_xlabel('Counts')
    axes[0].set_ylabel(r'$\mathrm{I_{cen}} \, / \langle \mathrm{I_{vic}} \rangle$', fontweight='bold')
    axes[0].set_ylim(0.94, 1.06)

    axes[-1].legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1.0), 
                    frameon=False, fontsize=15)

    fig.subplots_adjust(wspace=0.0, right=0.78, left=0.08, bottom=0.12, top=0.88) 
    plt.show()

vicinity_configs = [
    ('4', relative_vicinity_4, min4),
    ('8', relative_vicinity_8, min8),
    ('14', relative_vicinity_14, min14)
]

plot_vicinity_comparison(roi, lu_mask, vicinity_configs)

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

"""
Overlays tiered detection boxes onto the ROI based on how many metrics 
(Vicinity 4, 8, or 14) flagged a potential defect.
"""

roi_plotter = roi_tools.ROIPlotter(roi, title=None)

# Calculate detection counts across all three metrics
mask_sum = higher_map_4.astype(int) + higher_map_8.astype(int) + higher_map_14.astype(int)

# Plot Tier 1: Detected by exactly 1 metric (High transparency)
roi_plotter.add_boolean_patches_overlay(mask_sum == 1, color="#FFCFD9", alpha=0.6)

# Plot Tier 2: Detected by exactly 2 metrics (Medium transparency)
roi_plotter.add_boolean_patches_overlay(mask_sum == 2, color="#BB5D75", alpha=0.6)

# Plot Tier 3: Detected by all 3 metrics (No alpha/Opaque)
roi_plotter.add_boolean_patches_overlay(mask_sum == 3, color="#820021", alpha=0.6)

# Create tiered legend proxies to explain the alpha levels
tier1_proxy = mpatches.Patch(color='#FFCFD9', alpha=0.6, label='1 Metric', edgecolor='none')
tier2_proxy = mpatches.Patch(color='#BB5D75', alpha=0.6, label='2 Metrics', edgecolor='none')
tier3_proxy = mpatches.Patch(color='#820021', alpha=0.6, label='3 Metrics', edgecolor='none')

roi_plotter.ax.legend(
    handles=[tier1_proxy, tier2_proxy, tier3_proxy], 
    loc='upper left', 
    bbox_to_anchor=(1.01, 1.0), 
    frameon=False, 
    fontsize=15,
    title="Metrics Met",
    title_fontsize=15,
    handletextpad=0.5,
    borderaxespad=0
)

# Adjust margin for the multi-tier legend
roi_plotter.fig.subplots_adjust(right=0.8)

# roi_plotter.show(axis_on=False, scale_nm=2, scale_linewidth=5, scale_fontsize=12)
roi_plotter.show(axis_on=False, scale_nm=1, scale_linewidth=5, scale_fontsize=12)

In [ ]:
"""
Calculates and prints the number of defect atoms flagged by 1, 2, or all 3 metrics,
leveraging the combined mask_sum array.
"""
tier_1_count = np.sum(mask_sum == 1)
tier_2_count = np.sum(mask_sum == 2)
tier_3_count = np.sum(mask_sum == 3)

# Total unique defects (the inclusive approach we discussed)
total_unique_defects = tier_1_count + tier_2_count + tier_3_count

print("Defect Detection Breakdown:")
print(f"  Detected by exactly 1 metric:  {tier_1_count}")
print(f"  Detected by exactly 2 metrics: {tier_2_count}")
print(f"  Detected by all 3 metrics:     {tier_3_count} (High-Confidence Consensus)")
print("-" * 55)
print(f"  Total Unique Defect Atoms:     {total_unique_defects} (Inclusive Maximum)")

In [ ]:
import numpy as np

def count_consensus_defects(map_4, map_8, map_14):
    """
    Counts defects that are detected by all three metrics simultaneously.
    """
    # Create a mask where all three maps have a detection
    consensus_mask = (map_4 > 0) & (map_8 > 0) & (map_14 > 0)
    
    total_consensus = np.sum(consensus_mask)
    
    print(f"Total consensus defects: {total_consensus}")
    return total_consensus

# Counting the intersection using your provided variables
consensus_count = count_consensus_defects(higher_map_4, higher_map_8, higher_map_14)

In [ ]:
import numpy as np

def count_valid_entries(data_array, mask):
    """
    Counts the number of elements that are both non-NaN in the data_array 
    and True in the provided boolean mask.
    """
    valid_data_mask = ~np.isnan(data_array)
    combined_mask = valid_data_mask & mask
    
    return np.sum(combined_mask)

valid_count = count_valid_entries(roi.relative_vicinity, lu_mask)
print(f"Number of valid entries: {valid_count}")